# 🧬 Biohub Cell Tracking — 3D U-Net (2 models)

Inference/submission kernel: loads 2 trained 3D U-Nets, predicts a per-voxel cell-centre heatmap for every
frame (heatmaps AVERAGED — each model with its OWN preprocessing), turns peaks into detections, links them across time (two-pass µm-gated Hungarian),
writes `submission.csv`.
Weights = **`unet3d_bright.pt` (preproc: none), `unet3d_traintophat.pt` (preproc: tophat)** from the attached
`biohub-unet3d-weights` dataset; peak threshold **0.15**; own repair **True**.
Fully offline; conv3d GPU probe → CPU fallback (prefer T4 x2).

In [ ]:

import os, json, glob, time, gc
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from skimage.feature import peak_local_max

DEVICE = "cpu"
if torch.cuda.is_available():
    try:
        _p = nn.Conv3d(1,1,3).to("cuda"); _ = _p(torch.zeros(1,1,4,4,4,device="cuda")).cpu(); DEVICE="cuda"; del _p
    except Exception as e:
        print("GPU present but conv3d unusable (P100/sm_60?) -> CPU:", str(e)[:80])
SCALE = np.array([1.625, 0.40625, 0.40625]); POOL = 4
WEIGHT_NAMES = ['unet3d_bright.pt', 'unet3d_traintophat.pt']
PREPROCS = ['', 'tophat']          # one per model; a model MUST use the preproc it was trained with
REPAIR = True
CAND_THR = 0.05
GAP_DT = 0
GAP_GATE_UM = 10.0
SNAP_UM = 3.0
SHORT_MIN = 4
LINEFIT_WEIGHT = 0.8
LINEFIT_WINDOW = 2
print("device:", DEVICE, "| torch", torch.__version__)
print("models:", list(zip(WEIGHT_NAMES, [p or "none" for p in PREPROCS])))
print("repair:", REPAIR, "| seed_thr:", 0.15, "| cand_thr:", CAND_THR, "| gap_dt:", GAP_DT, "| short:", SHORT_MIN)

CANDIROOT = ["/kaggle/input/biohub-cell-tracking-during-development",
             "/kaggle/input/competitions/biohub-cell-tracking-during-development", "data"]
ROOT = next((p for p in CANDIROOT if Path(p,"test").exists()), "data"); TEST_DIR = Path(ROOT)/"test"
def _find_weight(name):
    cands = [f"/kaggle/input/biohub-unet3d-weights/{name}", f"models/{name}"] +             glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    p = next((c for c in cands if Path(c).exists()), None)
    if p is None:
        raise FileNotFoundError(f"{name} not found; attach biohub-unet3d-weights. "
                                f"/kaggle/input: {glob.glob('/kaggle/input/*')}")
    return p
WEIGHTS = [_find_weight(n) for n in WEIGHT_NAMES]
OUT = "submission.csv"; print("data:", ROOT, "| weights:", WEIGHTS)

def _block(ci, co):
    return nn.Sequential(nn.Conv3d(ci,co,3,padding=1), nn.BatchNorm3d(co), nn.ReLU(inplace=True),
                         nn.Conv3d(co,co,3,padding=1), nn.BatchNorm3d(co), nn.ReLU(inplace=True))
class UNet3D(nn.Module):
    def __init__(self, base=24):
        super().__init__()
        self.e1=_block(1,base); self.e2=_block(base,base*2); self.e3=_block(base*2,base*4)
        self.pool=nn.MaxPool3d(2); self.bott=_block(base*4,base*8)
        self.u3=nn.ConvTranspose3d(base*8,base*4,2,stride=2); self.d3=_block(base*8,base*4)
        self.u2=nn.ConvTranspose3d(base*4,base*2,2,stride=2); self.d2=_block(base*4,base*2)
        self.u1=nn.ConvTranspose3d(base*2,base,2,stride=2); self.d1=_block(base*2,base)
        self.out=nn.Conv3d(base,1,1)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.pool(e1)); e3=self.e3(self.pool(e2)); b=self.bott(self.pool(e3))
        d3=self.d3(torch.cat([self.u3(b),e3],1)); d2=self.d2(torch.cat([self.u2(d3),e2],1))
        d1=self.d1(torch.cat([self.u1(d2),e1],1)); return self.out(d1)

MODELS = []
for _w in WEIGHTS:
    _ck = torch.load(_w, map_location=DEVICE)
    _m = UNet3D(base=_ck.get("base",24)).to(DEVICE); _m.load_state_dict(_ck["state_dict"]); _m.eval()
    MODELS.append(_m)
    print("loaded", Path(_w).name, "| val_recall", _ck.get("val_recall"), "| aug", _ck.get("aug"))
assert len(MODELS)==len(PREPROCS), (len(MODELS), len(PREPROCS))


In [ ]:

def read_array_meta(zp):
    with open(Path(zp)/"0"/"zarr.json") as f: m=json.load(f)
    return dict(shape=tuple(m["shape"]), dtype=np.dtype(m["data_type"]))
_ZC={}
def load_volume(zp, t, meta=None):
    try:
        import zarr; k=str(zp)
        if k not in _ZC: _ZC[k]=zarr.open(k,mode="r")["0"]
        return np.asarray(_ZC[k][t])
    except Exception:
        import blosc2
        if meta is None: meta=read_array_meta(zp)
        buf=blosc2.decompress(open(Path(zp)/"0"/"c"/str(t)/"0"/"0"/"0","rb").read())
        return np.frombuffer(buf,dtype=meta["dtype"]).reshape(meta["shape"][1:])

def pool_xy(vol, f=POOL):
    Z,Y,X=vol.shape; Y2,X2=(Y//f)*f,(X//f)*f
    v=vol[:,:Y2,:X2].astype(np.float32,copy=False)
    return v.reshape(Z,Y2//f,f,X2//f,f).mean(axis=(2,4))
def pool_norm(vol, preproc=""):
    p=pool_xy(vol)
    if preproc=="tophat":
        from scipy.ndimage import grey_opening
        p=np.clip(p-grey_opening(p,size=(1,7,7)),0.0,None)
    lo=float(np.percentile(p,50)); hi=float(np.percentile(p,99.5))
    return np.clip((p-lo)/(hi-lo+1e-6),-0.5,6.0).astype(np.float32)

def _refine(vol, zyx, rz=2, ryx=5):
    Z,Y,X=vol.shape; z,y,x=(int(round(v)) for v in zyx)
    z0,z1=max(0,z-rz),min(Z,z+rz+1); y0,y1=max(0,y-ryx),min(Y,y+ryx+1); x0,x1=max(0,x-ryx),min(X,x+ryx+1)
    crop=vol[z0:z1,y0:y1,x0:x1].astype(np.float32); bg=float(crop.min())
    w=np.clip(crop-bg,0,None); s=float(w.sum())
    if s<=0: return np.array([z,y,x],float),0.0
    zz,yy,xx=np.mgrid[z0:z1,y0:y1,x0:x1]
    return np.array([(zz*w).sum(),(yy*w).sum(),(xx*w).sum()])/s, float(crop.max()-bg)
def _physical_nms(coords, scores, radius_um, scale=SCALE):
    if len(coords)<=1: return coords,scores
    pts=coords*scale[None,:]; order=np.argsort(-scores); tree=cKDTree(pts)
    killed=np.zeros(len(coords),bool); keep=[]
    for i in order:
        if killed[i]: continue
        keep.append(int(i)); killed[tree.query_ball_point(pts[i],r=radius_um)]=True
    keep=np.array(keep); return coords[keep],scores[keep]

UNET_THRESH=0.15; NMS_UM=4.0
DETECT_THRESH = min(UNET_THRESH, CAND_THR) if REPAIR else UNET_THRESH
def detect(vol):
    # each model sees the preprocessing it was TRAINED with, then AVERAGE the heatmaps
    # (never union the detections: DoG-union U-Net measured 0.661 -- over-detection
    #  blows past T_true and the node-count adjustment punishes it)
    hs=[]
    for _m,_pp in zip(MODELS, PREPROCS):
        x=pool_norm(vol,_pp)
        with torch.no_grad():
            hs.append(torch.sigmoid(_m(torch.from_numpy(x)[None,None].to(DEVICE)))[0,0].float().cpu().numpy())
    h = hs[0] if len(hs)==1 else np.mean(hs, axis=0)
    pk=peak_local_max(h, min_distance=1, threshold_abs=DETECT_THRESH, exclude_border=False)
    if len(pk)==0: return np.zeros((0,3)), np.zeros(0)
    sc=h[pk[:,0],pk[:,1],pk[:,2]].astype(float)
    coords=pk.astype(float); coords[:,1]=coords[:,1]*POOL+(POOL-1)/2; coords[:,2]=coords[:,2]*POOL+(POOL-1)/2
    ref=np.array([_refine(vol,c)[0] for c in coords])
    return _physical_nms(ref, sc, NMS_UM)


In [ ]:

MAX_LINK_UM=10.0; TIGHT_UM=6.0
def _link(prev_xyz, curr_xyz, prev_vel):
    if len(prev_xyz)==0 or len(curr_xyz)==0: return []
    P=prev_xyz*SCALE[None,:]; C=curr_xyz*SCALE[None,:]
    pred=P+(0.5*prev_vel if prev_vel is not None else 0.0); N,M=len(P),len(C); BIG=1e9
    def _hun(pi,ci,gate):
        if len(pi)==0 or len(ci)==0: return []
        Draw=np.sqrt(((P[pi][:,None]-C[ci][None])**2).sum(2)); D=np.sqrt(((pred[pi][:,None]-C[ci][None])**2).sum(2))
        cost=np.where(Draw>gate,BIG,D); ri,rc=linear_sum_assignment(cost)
        return [(int(pi[r]),int(ci[c])) for r,c in zip(ri,rc) if cost[r,c]<BIG]
    links=_hun(np.arange(N),np.arange(M),min(TIGHT_UM,MAX_LINK_UM))
    up={p for p,_ in links}; uc={c for _,c in links}
    fp=np.array([i for i in range(N) if i not in up],int); fc=np.array([j for j in range(M) if j not in uc],int)
    return links+_hun(fp,fc,MAX_LINK_UM)

COLS=["dataset","row_type","node_id","t","z","y","x","source_id","target_id"]
def _dist_um(a,b):
    d=(np.asarray(a,float)-np.asarray(b,float))*SCALE
    return float(np.sqrt((d*d).sum()))

def _gap_support(pe, ps, te, dt, cand, cand_trees):
    out=[]
    for k in range(1, dt):
        tk=te+k; interp=pe+(ps-pe)*(k/dt); tree=cand_trees[tk] if 0 <= tk < len(cand_trees) else None
        if tree is None: return None
        dist,idx=tree.query(interp*SCALE)
        if dist > SNAP_UM: return None
        out.append(cand[tk][idx])
    return out

def _segments(nodes, succ, pred, vel):
    segs=[]
    for g in nodes:
        if g in pred: continue
        ch=[g]
        while ch[-1] in succ: ch.append(succ[ch[-1]])
        segs.append(ch)
    ends=[]; starts=[]
    for ch in segs:
        ge=ch[-1]; ve=vel.get(ge, np.zeros(3))
        if len(ch) >= 2: ve=(nodes[ch[-1]]["xyz"]-nodes[ch[-2]]["xyz"])*SCALE
        ends.append((ge, int(nodes[ge]["t"]), nodes[ge]["xyz"], ve))
        gs=ch[0]; starts.append((gs, int(nodes[gs]["t"]), nodes[gs]["xyz"]))
    return segs, ends, starts

def _gap_close(nodes, edges, succ, pred, vel, cand, cand_trees, next_id):
    if GAP_DT <= 0: return next_id
    segs, ends, starts = _segments(nodes, succ, pred, vel)
    seglen=[len(ch) for ch in segs]; props=[]
    for i,(ge,te,pe,ve_um) in enumerate(ends):
        for j,(gs,ts,ps) in enumerate(starts):
            dt=ts-te
            if i == j or dt < 1 or dt > GAP_DT: continue
            predpos=pe+(ve_um/SCALE)*dt
            cost=_dist_um(predpos, ps)
            if cost > GAP_GATE_UM: continue
            if dt >= 2 and _gap_support(pe, ps, te, dt, cand, cand_trees) is None: continue
            props.append((cost,i,j,dt))
    used_e=set(); used_s=set()
    for _,i,j,dt in sorted(props):
        if i in used_e or j in used_s: continue
        used_e.add(i); used_s.add(j)
        ge,te,pe,_=ends[i]; gs,_,ps=starts[j]
        if dt == 1:
            edges.append((ge,gs)); continue
        prev=ge
        for k in range(1,dt):
            tk=te+k; interp=pe+(ps-pe)*(k/dt); use=interp
            if cand_trees[tk] is not None:
                dist,idx=cand_trees[tk].query(interp*SCALE)
                if dist <= SNAP_UM: use=cand[tk][idx]
            ng=next_id; next_id += 1
            nodes[ng]={"t":tk, "xyz":np.asarray(use,float)}
            edges.append((prev,ng)); prev=ng
        edges.append((prev,gs))
    return next_id

def _short_filter(nodes, edges):
    if SHORT_MIN <= 1 or not edges: return nodes, edges
    parent={nid:nid for nid in nodes}
    def find(x):
        while parent[x] != x:
            parent[x]=parent[parent[x]]; x=parent[x]
        return x
    def union(a,b):
        if a not in parent or b not in parent: return
        ra,rb=find(a),find(b)
        if ra != rb: parent[ra]=rb
    out_count=defaultdict(int)
    for a,b in edges:
        union(a,b); out_count[a]+=1
    comps=defaultdict(list)
    for nid in nodes: comps[find(nid)].append(nid)
    keep=set()
    for members in comps.values():
        has_div=any(out_count[n] >= 2 for n in members)
        if len(members) >= SHORT_MIN or has_div: keep.update(members)
    nodes2={nid:n for nid,n in nodes.items() if nid in keep}
    edges2=[(a,b) for a,b in edges if a in nodes2 and b in nodes2]
    return nodes2, edges2

def _linefit(nodes, edges):
    if LINEFIT_WEIGHT <= 0: return
    pred=defaultdict(list); succ=defaultdict(list)
    for a,b in edges:
        if a in nodes and b in nodes and int(nodes[b]["t"]) == int(nodes[a]["t"]) + 1:
            succ[a].append(b); pred[b].append(a)
    orig={k:v["xyz"].copy() for k,v in nodes.items()}; updates={}
    W=int(LINEFIT_WINDOW)
    for nid in nodes:
        neigh=[(0,nid)]; cur=nid
        for step in range(1,W+1):
            ps=pred.get(cur, [])
            if len(ps) != 1: break
            cur=ps[0]; neigh.append((-step,cur))
        cur=nid
        for step in range(1,W+1):
            ss=succ.get(cur, [])
            if len(ss) != 1: break
            cur=ss[0]; neigh.append((step,cur))
        if len(neigh) < 3: continue
        dt=np.array([a for a,_ in neigh], float)
        xyz=np.stack([orig[n] for _,n in neigh])
        fit=np.array([np.polyval(np.polyfit(dt, xyz[:,ax], 1), 0.0) for ax in range(3)])
        if np.isfinite(fit).all():
            updates[nid]=(1.0-LINEFIT_WEIGHT)*orig[nid]+LINEFIT_WEIGHT*fit
    for nid,xyz in updates.items(): nodes[nid]["xyz"]=xyz

def _emit(ds, nodes, edges):
    edge_set=[]; seen=set()
    for a,b in edges:
        if a == b or a not in nodes or b not in nodes or (a,b) in seen: continue
        seen.add((a,b)); edge_set.append((a,b))
    used=set()
    for a,b in edge_set: used.add(a); used.add(b)
    nrows=[]; erows=[]
    for nid in sorted(used):
        n=nodes[nid]; z,y,x=n["xyz"]
        nrows.append((ds,"node",int(nid),int(n["t"]),float(z),float(y),float(x),-1,-1))
    for a,b in edge_set:
        if a in used and b in used: erows.append((ds,"edge",-1,-1,-1,-1,-1,int(a),int(b)))
    return pd.DataFrame(nrows,columns=COLS), pd.DataFrame(erows,columns=COLS)

def repair_track(dets, ds):
    nodes={}; frame_ids=[]; cand=[]; cand_trees=[]; nid=1
    for t,(coords,scores) in enumerate(dets):
        coords=np.asarray(coords,float).reshape(-1,3); scores=np.asarray(scores,float).reshape(-1)
        seeds=coords[scores >= UNET_THRESH]
        cands=coords[(scores >= CAND_THR) & (scores < UNET_THRESH)]
        cand.append(cands); cand_trees.append(cKDTree(cands*SCALE) if len(cands) else None)
        ids=[]
        for xyz in seeds:
            nodes[nid]={"t":t, "xyz":np.asarray(xyz,float)}; ids.append(nid); nid += 1
        frame_ids.append(ids)
    edges=[]; succ={}; pred={}; vel={}
    for t in range(len(dets)-1):
        P=np.asarray([nodes[g]["xyz"] for g in frame_ids[t]], float).reshape(-1,3)
        C=np.asarray([nodes[g]["xyz"] for g in frame_ids[t+1]], float).reshape(-1,3)
        if len(P) == 0 or len(C) == 0: continue
        prev_vel=np.array([vel.get(g, np.zeros(3)) for g in frame_ids[t]])
        for pi,ci in _link(P, C, prev_vel if len(prev_vel) else None):
            gp,gc=frame_ids[t][pi],frame_ids[t+1][ci]
            edges.append((gp,gc)); succ[gp]=gc; pred[gc]=gp; vel[gc]=(C[ci]-P[pi])*SCALE
    nid=_gap_close(nodes, edges, succ, pred, vel, cand, cand_trees, nid)
    nodes,edges=_short_filter(nodes, edges)
    _linefit(nodes, edges)
    return _emit(ds, nodes, edges)

def track_movie(zp, ds, T):
    if REPAIR:
        meta=read_array_meta(zp); dets=[]
        for t in range(T):
            dets.append(detect(load_volume(zp,t,meta))); gc.collect()
        return repair_track(dets, ds)
    meta=read_array_meta(zp); node_rows=[]; edge_rows=[]
    prev_ids=[]; prev_xyz=np.zeros((0,3)); prev_vel=None; nid=1
    for t in range(T):
        coords,scores=detect(load_volume(zp,t,meta)); gc.collect()
        ids=list(range(nid,nid+len(coords))); nid+=len(coords)
        for i,c in zip(ids,coords): node_rows.append((ds,"node",i,t,float(c[0]),float(c[1]),float(c[2]),-1,-1))
        if t>0 and len(prev_ids):
            links=_link(prev_xyz,coords,prev_vel); vel=np.zeros((len(prev_xyz),3))
            for p,c in links:
                edge_rows.append((ds,"edge",-1,-1,-1,-1,-1,prev_ids[p],ids[c])); vel[p]=(coords[c]-prev_xyz[p])*SCALE
            nv=np.zeros((len(coords),3))
            for p,c in links: nv[c]=vel[p]
            prev_vel=nv
        else: prev_vel=None
        prev_ids,prev_xyz=ids,coords
    nodes=pd.DataFrame(node_rows,columns=COLS); edges=pd.DataFrame(edge_rows,columns=COLS)
    if len(edges):
        used=set(edges.source_id)|set(edges.target_id); nodes=nodes[nodes.node_id.isin(used)].reset_index(drop=True)
    return nodes,edges

def avail_T(zp):
    meta=read_array_meta(zp); T=meta["shape"][0]
    present=[t for t in range(T) if (Path(zp)/"0"/"c"/str(t)/"0"/"0"/"0").exists()]
    return max(present)+1 if present else 0

parts=[]
for zp in sorted(TEST_DIR.glob("*.zarr")):
    ds=zp.name.replace(".zarr",""); T=avail_T(zp)
    if T==0: print("skip",ds); continue
    t0=time.time(); nodes,edges=track_movie(zp,ds,T); parts+=[nodes,edges]
    print(f"  {ds}: T={T} nodes={len(nodes)} edges={len(edges)} ({time.time()-t0:.1f}s)")
sub=pd.concat(parts,ignore_index=True); sub.index.name="id"; sub.to_csv(OUT)
exp=["dataset","row_type","node_id","t","z","y","x","source_id","target_id"]; assert list(sub.columns)==exp
print("wrote",OUT,"rows",len(sub),"| nodes",(sub.row_type=='node').sum(),"edges",(sub.row_type=='edge').sum())
